# Fraud Detection: Power BI Data Export

## Goal
Export cleaned and model-scored data for the Power BI dashboard.
This notebook generates 4 CSV files used in the dashboard:

1. **transactions.csv**: full dataset with model predictions
2. **fraud_by_hour.csv**: fraud rate aggregated by hour
3. **fraud_by_bucket.csv**:  fraud losses by transaction size
4. **model_performance.csv**:  precision, recall, F1 for all 3 models

## Note
The Power BI dashboard uses the full dataset (284,807 transactions)
while the modeling notebook evaluates on the test set only (56,962 
transactions). Both are valid modeling shows evaluation results,
and the dashboard shows the full business impact.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

# Load data
df_original = pd.read_csv('/Users/mac/creditcard.csv')
df = df_original.copy()

# Preprocessing
df['Amount_Scaled'] = StandardScaler().fit_transform(df[['Amount']])
df['Time_Scaled'] = StandardScaler().fit_transform(df[['Time']])
df['Hour'] = (df['Time'] / 3600).astype(int) % 24
df = df.drop(['Amount', 'Time'], axis=1)

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_sm, y_train_sm)

print('Ready to export!')

In [ ]:
import os
os.makedirs('/Users/mac/fraud-detection/powerbi_data', exist_ok=True)

# 1. Full dataset with predictions
df_export = df_original.copy()
df_export['Hour'] = (df_export['Time'] / 3600).astype(int) % 24
df_export['Predicted'] = rf_model.predict(
    df.drop('Class', axis=1)
)
df_export.to_csv('/Users/mac/fraud-detection/powerbi_data/transactions.csv', index=False)

print('✅ Data exported for Power BI!')

In [ ]:
import os

path = '/Users/mac/fraud-detection/powerbi_data'
print(os.path.exists(path))
print(os.listdir(path))

In [ ]:
# Fraud rate by hour
hour_summary = df_export.groupby('Hour').agg(
    total=('Class', 'count'),
    fraud=('Class', 'sum')
).reset_index()
hour_summary['fraud_rate'] = (hour_summary['fraud'] / hour_summary['total'] * 100).round(3)
hour_summary.to_csv('/Users/mac/fraud-detection/powerbi_data/fraud_by_hour.csv', index=False)

# Fraud by amount bucket
df_export['AmountBucket'] = pd.cut(df_export['Amount'],
    bins=[0, 10, 50, 100, 500, 5000, df_export['Amount'].max()],
    labels=['<$10','$10-50','$50-100','$100-500','$500-5k','>$5k'])
bucket_summary = df_export.groupby('AmountBucket', observed=True).agg(
    total=('Class', 'count'),
    fraud=('Class', 'sum'),
    total_loss=('Amount', lambda x: x[df_export.loc[x.index, 'Class']==1].sum())
).reset_index()
bucket_summary.to_csv('/Users/mac/fraud-detection/powerbi_data/fraud_by_bucket.csv', index=False)

# Model performance summary
model_perf = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Precision': [0.06, 0.82, 0.73],
    'Recall': [0.92, 0.82, 0.89],
    'F1': [0.11, 0.82, 0.80]
})
model_perf.to_csv('/Users/mac/fraud-detection/powerbi_data/model_performance.csv', index=False)

print('✅ All files exported!')
print(os.listdir('/Users/mac/fraud-detection/powerbi_data'))